# Fitcast — 날씨·체형·취향을 함께 읽는 코디 예보 AI 도우미

**LangChain 종합실습 · 최인서**

> 이 노트북은 서비스 **Fitcast OOTD**(웹 아바타 피팅룸)에서 **LLM · LangChain 부분만** 떼어 설명·시연합니다.
> 전체 서비스 코드는 같은 저장소의 `fitcast/` 패키지에 있고, 노트북은 그 모듈을 그대로 import해서 실행합니다.

---

## 1. 주제 선정 — 왜 LLM이어야 했는가

**한 문장 정의**
"아침에 일기예보는 봤는데 뭘 입을지 모르겠는 사람"에게, **오늘 날씨·내 체형·좋아하는 스타일·TPO**를 한 번에 반영한 **코디 한 벌**을 아이템별 이유와 함께 예보해 주는 도우미.

**규칙·검색으로 안 되는 이유**

| 방법 | 한계 |
|---|---|
| 규칙 기반 (기온 구간표) | "23°C면 반팔"까지는 되지만, *모리걸 + 결혼식 하객 + 비 + 하체 중심 체형*처럼 조건이 **곱해지는 순간 규칙 수가 폭발**한다. 소재·기장·레이어드 수를 조건에 맞게 **조절**하는 판단은 규칙으로 못 쓴다. |
| 단순 검색 | "여름 코디" 검색 결과는 내 체형·TPO·오늘 강수확률을 모른다. 결과를 **한 벌로 조합**하고 **이유를 설명**해 주지 않는다. |
| **LLM** | 날씨 수치(정형) + 스타일 가이드(비정형 문서) + 사용자 취향(자연어)을 **한 컨텍스트에 놓고** 조합·조절·설명할 수 있다. 다만 LLM은 오늘 날씨를 모르고 브랜드·가격을 지어내므로, **Tool(날씨 API)·RAG(가이드 문서)·구조화 출력(스키마)** 으로 컨텍스트를 채우고 출력을 묶어야 한다. |

즉 이 도우미의 핵심은 **"모델에 무엇을 어떤 형태로 넣는가"** 입니다. 아래에서 그 컨텍스트가 어떻게 만들어지는지 따라갑니다.

## 발표 순서 (5분) · 테스트 시나리오

| 시간 | 내용 | 셀 |
|---|---|---|
| 0:00 | 기획 의도 — 누구의 불편, 왜 LLM인가 (핵심만) | 1장 |
| 0:45 | 시연 A: 입력 하나가 날씨 Tool → 컨텍스트 → 프롬프트 → 구조화 출력 → 마크다운으로 흐르는 과정 | 2장 ①~④ |
| 2:15 | 시연 B·C: 도시·TPO·체형만 바꿔 결과가 어떻게 달라지는지 비교표 | 2장 비교 |
| 3:15 | 컴포넌트 — 프롬프트·LCEL·구조화 출력·RAG·Tool/Agent·메모리, 없었다면? | 3장 |
| 4:15 | 한계와 개선 방향 | 4장 |

**테스트 시나리오 (노트북에 저장된 실행 결과 기준, API가 느리면 저장 결과로 설명)**

| # | 입력 | 확인할 것 |
|---|---|---|
| A | 서울 · 내일 · 모리걸 · 일상 · 154cm/48kg 삼각형 체형 | 날씨 수치·기온 가이드·RAG 조각이 human 메시지에 들어가고, 스키마대로 5개 슬롯 + 팁이 나오는지 |
| B | A에서 도시 제주 · TPO 여행 | 날씨가 비슷하면 아이템은 유지되고 팁·이유에 '여행'이 반영되는지 |
| C | A에서 미니멀 · 출근/오피스 · 168cm/56kg 역삼각형 체형 | TPO 규칙(반바지·민소매 금지)과 체형 규칙이 아이템·이유에 드러나는지 |
| 에이전트 1턴 | "이번 주 토요일에 제주도 여행 가는데 미니멀하게 뭐 입을까요?" | 날짜를 계산해 날씨 Tool을 스스로 부르는지 |
| 에이전트 2턴 | "신발만 좀 더 편한 걸로 바꿔줘" | 이전 코디를 기억해 신발만 바꾸는지 (메모리) |
| 에이전트 3턴 | "파이썬으로 피보나치 함수 짜줘" | 코디와 무관한 요청을 규칙대로 거절하는지 |

**실행 방법**: 저장소 루트에서 `pip install -r requirements.txt` 후 `.env`에 `OPENAI_API_KEY`를 넣고, 이 노트북을 루트(또는 한 단계 아래)에서 엽니다. 날씨는 Open-Meteo(무료, 키 불필요)를 씁니다.


### 0. 준비 — 프로젝트 모듈 불러오기

`.env`의 `OPENAI_API_KEY`를 읽습니다(키 값은 출력하지 않습니다). 모델은 `FITCAST_MODEL`(기본 `openai:gpt-4o-mini`).

In [1]:
import sys, json, os
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "fitcast").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from IPython.display import Markdown, display
from fitcast import config

print("모델:", config.MODEL_NAME, "| 임베딩:", config.EMBEDDING_MODEL)
print("OPENAI_API_KEY 설정:", bool(os.getenv("OPENAI_API_KEY")))

모델: openai:gpt-4o-mini | 임베딩: openai:text-embedding-3-small
OPENAI_API_KEY 설정: True


---

## 2. 문제 해결 — 입력 하나를 끝까지 따라가기

체인 한 번 호출은 아래 5단계로 흐릅니다.

```
입력 dict ─▶ ① 날씨 Tool 호출 ─▶ ② 컨텍스트 구성(기온 가이드 + RAG 검색, 병렬) ─▶ ③ 프롬프트 → LLM(구조화 출력) ─▶ ④ 후처리(마크다운/아바타 매핑) ─▶ 출력
```

시연 입력: **서울 · 내일 · 모리걸 · 일상 · 154cm/48kg 삼각형(하체 중심) 체형**

In [2]:
from datetime import date, timedelta

TOMORROW = (date.today() + timedelta(days=1)).isoformat()

user_input = {
    "city": "서울",
    "date": TOMORROW,
    "styles": ["모리걸"],
    "tpo": "일상",
    "gender": "여성",
    "note": "많이 걸을 예정",
    "profile": "154cm / 48kg (BMI 20.2), 체형 삼각형(하체 중심), 헤어 롱 웨이브, 얼굴 분위기 강아지상",
}
user_input

{'city': '서울',
 'date': '2026-09-23',
 'styles': ['모리걸'],
 'tpo': '일상',
 'gender': '여성',
 'note': '많이 걸을 예정',
 'profile': '154cm / 48kg (BMI 20.2), 체형 삼각형(하체 중심), 헤어 롱 웨이브, 얼굴 분위기 강아지상'}

### ① 날씨 Tool — LLM이 모르는 '오늘'을 컨텍스트로

Open-Meteo(무료, 키 불필요)를 부르는 `@tool get_weather`의 내부 함수입니다. 체인에서는 함수로 직접 호출하고, 챗봇 에이전트에서는 같은 것을 Tool로 호출합니다.

In [3]:
from fitcast.tools.weather import fetch_weather, format_weather

weather_data = fetch_weather(user_input["city"], user_input["date"])
print(json.dumps(weather_data, ensure_ascii=False, indent=1))
print()
print("LLM에 들어가는 문장 →", format_weather(weather_data))

{
 "city": "서울",
 "date": "2026-09-23",
 "condition": "구름 조금",
 "temp_max": 26.2,
 "temp_min": 18.0,
 "temp_avg": 22.1,
 "feels_max": 28.1,
 "feels_min": 19.6,
 "rain_prob": 20,
 "rain_mm": 0.0,
 "wind_kmh": 5.1,
 "uv_index": 6.0
}

LLM에 들어가는 문장 → [서울 2026-09-23] 구름 조금, 최고 26.2°C / 최저 18.0°C (체감 28.1°C / 19.6°C), 강수확률 20%, 강수량 0.0mm, 최대 풍속 5.1km/h, 자외선 지수 6.0


### ② 컨텍스트 구성 — 규칙(기온 구간표) + RAG(스타일 가이드)

- `temp_band_guide`: 평균 기온으로 **규칙 기반** 구간표(`data/temp_guide.json`)에서 대표 아이템을 뽑습니다. 규칙으로 확실한 부분은 규칙으로 넣어 LLM의 실수를 줄입니다.
- `retrieve_style_context`: `data/style_guide.md`를 `## 섹션` 단위로 쪼개 임베딩한 **InMemoryVectorStore**에서, "스타일 + 더운/추운 날 + TPO" 질의로 가까운 조각 3개를 가져옵니다.

In [4]:
from fitcast.rag.style_guide import temp_band_guide, retrieve_style_context
from fitcast.chains.outfit import _style_query

x = {**user_input, "styles": ", ".join(user_input["styles"]), "weather_data": weather_data}
temp_guide = temp_band_guide(weather_data["temp_avg"])
query = _style_query(x)
style_context = retrieve_style_context(query)

print("기온 구간 가이드 →", temp_guide)
print("\nRAG 질의 →", query)
print("\nRAG 검색 결과(상위 3조각) ↓\n")
print(style_context)

기온 구간 가이드 → 늦봄·초가을 (20~22°C): 대표 아이템 얇은 가디건, 긴팔 티, 블라우스, 면바지, 청바지, 롱스커트. 긴팔 한 장 또는 반팔+가디건 레이어드

RAG 질의 → 모리걸 선선한 날 / TPO: 일상 / 날씨 보정 규칙

RAG 검색 결과(상위 3조각) ↓

## TPO: 면접
네이비·차콜·블랙 셋업에 흰 셔츠 또는 단정한 블라우스. 악세사리는 최소화. 한여름엔 재킷을 들고 가서 입장 전에 입는다.

## 모리걸
'숲속 소녀' 콘셉트. 아이보리·베이지·카키·브라운 같은 자연색, 린넨·코튼·니트 같은 천연 소재, 여러 겹의 헐렁한 레이어드와 레이스·자수·잔꽃무늬 디테일이 특징.
더운 날(25°C 이상): 레이어드를 줄이고 소재로 분위기를 낸다. 얇은 린넨 롱 원피스 한 장, 또는 코튼 레이스 블라우스 + 린넨 롱스커트. 밀짚모자(라피아햇), 라탄백, 가죽 스트랩 샌들. 실내 냉방 대비로 크로셰 볼레로나 얇은 가디건.
선선한 날: 원피스 위에 니트 베스트나 롱 가디건, 레이스 양말 + 메리제인 또는 워커.
추운 날: 울 롱스커트 + 두꺼운 케이블 니트 + 더플코트, 니트 머플러와 베레모.

## TPO: 여행
많이 걷는 걸 전제로 편한 신발 우선. 일교차와 현지 실내 냉난방에 대비한 얇은 겉옷, 구김 적은 소재. 사진에 잘 나오는 포인트 컬러 하나.


### ③ 프롬프트 → LLM (구조화 출력)

세 재료가 채워진 **실제 프롬프트**를 먼저 보고, 그다음 체인을 실행합니다.

In [5]:
from fitcast.prompts import OUTFIT_PROMPT
from fitcast.chains.outfit import shape_guide

filled = OUTFIT_PROMPT.format_messages(
    weather=format_weather(weather_data), temp_guide=temp_guide, style_context=style_context,
    styles=", ".join(user_input["styles"]), tpo=user_input["tpo"], gender=user_input["gender"],
    profile=user_input["profile"], note=user_input["note"], shape_guide=shape_guide(),
)
for m in filled:
    print(f"===== {m.type.upper()} =====")
    print(m.content[:1800] + ("\n... (생략)" if len(m.content) > 1800 else ""))
    print()

===== SYSTEM =====
너는 날씨와 개인 취향을 함께 고려하는 퍼스널 스타일리스트 'Fitcast'야.

규칙:
1. 날씨의 기온, 체감온도, 강수, 바람, 자외선을 최우선으로 고려한다.
2. 선호 스타일의 분위기를 유지하되, 날씨에 따라 소재, 기장, 두께, 레이어드 수를 조절한다.
3. TPO와 드레스코드에 어긋나는 아이템은 추천하지 않는다.
4. search_keyword는 쇼핑 검색에 바로 사용할 수 있는 짧은 2~3단어 한국어로 작성한다.
5. search_keyword에 브랜드명은 넣지 않는다.
6. 특정 브랜드명, 상품명, 가격, 재고는 지어내지 않는다.
7. 모든 문장은 친근한 한국어 존댓말로 작성한다.
8. 체형 정보가 있으면 체형을 자연스럽게 보완하는 핏과 기장을 선택한다.
9. 체형을 고려한 이유를 각 주요 아이템 설명에 한 번씩 포함한다.
10. shape는 아래 목록에서 해당 부위의 코드만 사용한다.
11. color_hex는 추천 색상과 가까운 실제 HEX 값으로 작성한다.

## shape 코드 가이드
- top: tee, shirt, blouse, knit, cami, hoodie, sweat, crop, jersey, turtleneck, polo, sleeveless
- bottom: wide, straight, bootcut, slacks, cargo, jogger, shorts, leggings, mini, pleats, midi, long_skirt, slip_dress, long_dress, knit_dress
- outer: cardigan, bolero, blazer, trench, coat, padding, leather, windbreaker, jacket, knit_vest
- shoes: sneakers, loafers, boots, combat, long_boots, mules, sandals, maryjane, flats, heels
- accessory: bag_shoulder, bag_tote

In [6]:
from fitcast.chains.outfit import recommend_outfit

result = recommend_outfit(**user_input)
outfit = result["outfit"]           # Pydantic 모델 OutfitSet
print(type(outfit).__name__)
print(json.dumps(outfit.model_dump(), ensure_ascii=False, indent=1))

OutfitSet
{
 "summary": "오늘의 모리걸 스타일 코디입니다!",
 "top": {
  "name": "린넨 블라우스",
  "reason": "얇은 린넨 소재로 통기성이 좋고, 자연스러운 분위기를 연출해 체형을 보완합니다.",
  "search_keyword": "린넨 블라우스",
  "shape": "blouse",
  "color_hex": "#f5f5dc"
 },
 "bottom": {
  "name": "롱 린넨 스커트",
  "reason": "하체 중심의 삼각형 체형을 보완하며, 편안한 착용감으로 많이 걷는 데 적합합니다.",
  "search_keyword": "롱 린넨 스커트",
  "shape": "long_skirt",
  "color_hex": "#d2b48c"
 },
 "outer": {
  "name": "얇은 가디건",
  "reason": "선선한 날씨에 적합하며, 레이어드로 스타일을 더해줍니다.",
  "search_keyword": "얇은 가디건",
  "shape": "cardigan",
  "color_hex": "#fff8dc"
 },
 "shoes": {
  "name": "가죽 스트랩 샌들",
  "reason": "편안하게 걸을 수 있는 신발로, 모리걸 스타일에 잘 어울립니다.",
  "search_keyword": "가죽 스트랩 샌들",
  "shape": "sandals",
  "color_hex": "#deb887"
 },
 "accessory": {
  "name": "라탄백",
  "reason": "자연스러운 느낌을 주며, 모리걸 스타일을 완성하는 아이템입니다.",
  "search_keyword": "라탄백",
  "shape": "bag_tote",
  "color_hex": "#f5deb3"
 },
 "weather_tip": "오늘은 기온이 따뜻하니, 얇은 소재의 옷을 선택하세요. 햇볕이 강하니 자외선 차단도 고려해주세요."
}


### ④ 후처리 → 출력

구조화 출력이라서 후처리는 필드를 꺼내 쓰는 일뿐입니다.
- 사람이 읽는 화면: `outfit_to_markdown` (아이템별 이유 + 쇼핑 검색 링크)
- 아바타 피팅룸: `shape`(모양 코드)·`color_hex`로 아바타에 옷을 그리고, `search_keyword`로 실제 상품을 검색

In [7]:
from fitcast.chains.outfit import outfit_to_markdown

display(Markdown(outfit_to_markdown(result, platforms=["무신사", "29CM"])))

print("\n아바타·상품 검색으로 넘어가는 값 ↓")
for slot in ("top", "bottom", "outer", "shoes", "accessory"):
    it = getattr(outfit, slot)
    if it:
        print(f"- {slot:9s} shape={it.shape:12s} color={it.color_hex}  검색어='{it.search_keyword}'")

**[서울 2026-09-23] 구름 조금, 최고 26.2°C / 최저 18.0°C (체감 28.1°C / 19.6°C), 강수확률 20%, 강수량 0.0mm, 최대 풍속 5.1km/h, 자외선 지수 6.0**

## 👗 오늘의 모리걸 스타일 코디입니다!

### 상의 · 린넨 블라우스
얇은 린넨 소재로 통기성이 좋고, 자연스러운 분위기를 연출해 체형을 보완합니다.

🔎 `린넨 블라우스` → [무신사](https://www.musinsa.com/search/goods?keyword=%EB%A6%B0%EB%84%A8%20%EB%B8%94%EB%9D%BC%EC%9A%B0%EC%8A%A4) · [29CM](https://www.29cm.co.kr/search?keyword=%EB%A6%B0%EB%84%A8%20%EB%B8%94%EB%9D%BC%EC%9A%B0%EC%8A%A4)


### 하의 · 롱 린넨 스커트
하체 중심의 삼각형 체형을 보완하며, 편안한 착용감으로 많이 걷는 데 적합합니다.

🔎 `롱 린넨 스커트` → [무신사](https://www.musinsa.com/search/goods?keyword=%EB%A1%B1%20%EB%A6%B0%EB%84%A8%20%EC%8A%A4%EC%BB%A4%ED%8A%B8) · [29CM](https://www.29cm.co.kr/search?keyword=%EB%A1%B1%20%EB%A6%B0%EB%84%A8%20%EC%8A%A4%EC%BB%A4%ED%8A%B8)


### 외투 · 얇은 가디건
선선한 날씨에 적합하며, 레이어드로 스타일을 더해줍니다.

🔎 `얇은 가디건` → [무신사](https://www.musinsa.com/search/goods?keyword=%EC%96%87%EC%9D%80%20%EA%B0%80%EB%94%94%EA%B1%B4) · [29CM](https://www.29cm.co.kr/search?keyword=%EC%96%87%EC%9D%80%20%EA%B0%80%EB%94%94%EA%B1%B4)


### 신발 · 가죽 스트랩 샌들
편안하게 걸을 수 있는 신발로, 모리걸 스타일에 잘 어울립니다.

🔎 `가죽 스트랩 샌들` → [무신사](https://www.musinsa.com/search/goods?keyword=%EA%B0%80%EC%A3%BD%20%EC%8A%A4%ED%8A%B8%EB%9E%A9%20%EC%83%8C%EB%93%A4) · [29CM](https://www.29cm.co.kr/search?keyword=%EA%B0%80%EC%A3%BD%20%EC%8A%A4%ED%8A%B8%EB%9E%A9%20%EC%83%8C%EB%93%A4)


### 악세사리 · 라탄백
자연스러운 느낌을 주며, 모리걸 스타일을 완성하는 아이템입니다.

🔎 `라탄백` → [무신사](https://www.musinsa.com/search/goods?keyword=%EB%9D%BC%ED%83%84%EB%B0%B1) · [29CM](https://www.29cm.co.kr/search?keyword=%EB%9D%BC%ED%83%84%EB%B0%B1)


> ☂️ 오늘은 기온이 따뜻하니, 얇은 소재의 옷을 선택하세요. 햇볕이 강하니 자외선 차단도 고려해주세요.


아바타·상품 검색으로 넘어가는 값 ↓
- top       shape=blouse       color=#f5f5dc  검색어='린넨 블라우스'
- bottom    shape=long_skirt   color=#d2b48c  검색어='롱 린넨 스커트'
- outer     shape=cardigan     color=#fff8dc  검색어='얇은 가디건'
- shoes     shape=sandals      color=#deb887  검색어='가죽 스트랩 샌들'
- accessory shape=bag_tote     color=#f5deb3  검색어='라탄백'


### 입력을 바꿔 비교 — 무엇이 달라지는가

같은 체인에 **날씨(도시)·TPO·체형**만 바꿔 두 번 더 실행합니다.

| 실행 | 도시 | TPO | 스타일 | 체형 |
|---|---|---|---|---|
| A (위) | 서울 | 일상 | 모리걸 | 삼각형(하체 중심) |
| B | 제주 | 여행 | 모리걸 | 삼각형(하체 중심) |
| C | 서울 | 출근/오피스 | 미니멀 | 역삼각형(어깨 넓음) |

기대: B는 **날씨(바람·강수)** 차이가 아이템에 반영되고, C는 **TPO 규칙**(반바지·민소매 금지)과 **체형 규칙**(어깨를 강조하지 않는 핏)이 이유 문장에 드러나야 합니다.

In [8]:
runs = {
    "A 서울·일상·모리걸": result,
    "B 제주·여행·모리걸": recommend_outfit(**{**user_input, "city": "제주", "tpo": "여행"}),
    "C 서울·오피스·미니멀": recommend_outfit(**{**user_input, "styles": ["미니멀"], "tpo": "출근/오피스",
                                                "profile": "168cm / 56kg (BMI 19.8), 체형 역삼각형(어깨 넓음), 헤어 단발, 얼굴 분위기 고양이상"}),
}

def row(label, r):
    o = r["outfit"]; w = r["weather_data"]
    cell = lambda it: f"{it.name}<br><small>{it.reason}</small>" if it else "—"
    return f"| **{label}** | {w['city']} {w['condition']} {w['temp_min']}~{w['temp_max']}°C · 강수 {w['rain_prob']}% | {cell(o.top)} | {cell(o.bottom)} | {cell(o.outer)} | {cell(o.shoes)} | {o.weather_tip} |"

table = "| 실행 | 날씨 | 상의 | 하의 | 외투 | 신발 | 날씨 팁 |\n|---|---|---|---|---|---|---|\n" + "\n".join(row(k, v) for k, v in runs.items())
display(Markdown(table))

| 실행 | 날씨 | 상의 | 하의 | 외투 | 신발 | 날씨 팁 |
|---|---|---|---|---|---|---|
| **A 서울·일상·모리걸** | 서울 구름 조금 18.0~26.2°C · 강수 20% | 린넨 블라우스<br><small>얇은 린넨 소재로 통기성이 좋고, 자연스러운 분위기를 연출해 체형을 보완합니다.</small> | 롱 린넨 스커트<br><small>하체 중심의 삼각형 체형을 보완하며, 편안한 착용감으로 많이 걷는 데 적합합니다.</small> | 얇은 가디건<br><small>선선한 날씨에 적합하며, 레이어드로 스타일을 더해줍니다.</small> | 가죽 스트랩 샌들<br><small>편안하게 걸을 수 있는 신발로, 모리걸 스타일에 잘 어울립니다.</small> | 오늘은 기온이 따뜻하니, 얇은 소재의 옷을 선택하세요. 햇볕이 강하니 자외선 차단도 고려해주세요. |
| **B 제주·여행·모리걸** | 제주 구름 조금 20.5~24.6°C · 강수 0% | 린넨 블라우스<br><small>얇은 린넨 소재로 통기성이 좋고 시원하여 더운 날씨에 적합하며, 부드러운 핏이 체형을 보완해 줍니다.</small> | 롱 린넨 스커트<br><small>롱 기장으로 하체를 가려주고, 자연스러운 컬러와 흐르는 실루엣이 삼각형 체형을 보완합니다.</small> | 얇은 니트 가디건<br><small>여름철 실내 냉방에 대비할 수 있는 얇은 소재로, 레이어드하기 좋으며, 모리걸 스타일에 잘 어울립니다.</small> | 가죽 스트랩 샌들<br><small>편안한 착용감과 보행에 적합한 디자인으로 여행 중 많이 걷는 상황에서 유용합니다.</small> | 자외선 지수가 높으니 자외선 차단제를 바르고, 모자나 선글라스를 착용하는 것이 좋습니다. |
| **C 서울·오피스·미니멀** | 서울 구름 조금 18.0~26.2°C · 강수 20% | 단정한 블라우스<br><small>블라우스는 고급스러운 분위기를 주며, 어깨가 넓은 체형을 보완해 줍니다.</small> | 슬랙스<br><small>슬랙스는 편안하면서도 세련된 느낌을 주어 출근 오피스룩에 적합하며, 다리 라인을 슬림하게 보완해줍니다.</small> | 얇은 가디건<br><small>가디건은 기온 변화에 대응하기 좋고, 미니멀한 스타일을 유지하는 데 도움을 줍니다.</small> | 플랫 슈즈<br><small>플랫 슈즈는 편안하면서도 깔끔한 인상을 주어 많이 걷는 일정에 알맞습니다.</small> | 오늘은 햇볕이 강하니 자외선 차단제를 꼭 바르세요! |

**비교에서 확인한 것 (실제 출력 기준)**

- **A → B (서울 일상 → 제주 여행)**: 날씨가 비슷해서(구름 조금, 20°C대, 강수 0~20%) 상의·하의·신발은 같은 아이템이 나왔다. 대신 `weather_tip`이 자외선·모자·선글라스로 바뀌고, 신발·외투 이유에 "여행 중 많이 걷는 상황", "실내 냉방 대비"가 들어갔다. → 날씨 컨텍스트(①)와 RAG의 `TPO: 여행` 섹션(②)이 **이유 문장과 팁**에 반영된 것. 날씨가 비슷하면 결과도 비슷하게 유지된다는 점은 안정성이지만, 다양성은 낮다.
- **A → C (모리걸 일상 → 미니멀 오피스, 역삼각형)**: 린넨 스커트·샌들이 **블라우스·슬랙스·플랫 슈즈**로 바뀌고, 상의 이유에 "어깨가 넓은 체형을 보완"이 들어갔다. → 프롬프트 규칙 3(TPO)과 8·9(체형)가 작동.

**아직 막힌 부분**
- ②의 RAG 결과를 보면 질의 "모리걸 선선한 날 / TPO: 일상"에 **`TPO: 면접` 조각이 1위**로 섞여 들어왔다. '일상'에 해당하는 TPO 섹션이 문서에 없어 가장 가까운 TPO 조각이 끌려온 것. 이번 결과에는 영향이 없었지만(모리걸 조각이 함께 들어가서), 점수 임계값이나 메타데이터 필터가 필요하다(4장).
- 같은 입력을 여러 번 돌리면 아이템 이름은 조금씩 달라진다(온도 기본값). `search_keyword`가 너무 구체적이면 실제 상품 검색이 0개일 때가 있어 서비스에서는 키워드를 짧게 잘라 재검색하는 폴백을 뒀다.

---

## 3. LangChain 컴포넌트 — 무엇을 어디에, 없었다면?

| 컴포넌트 | 위치 | 없었다면 |
|---|---|---|
| `ChatPromptTemplate` (system/human 분리, 변수 9개) | `fitcast/prompts.py` | 날씨·가이드·취향을 문자열 결합으로 끼워 넣어야 하고, 규칙과 데이터가 섞여 프롬프트를 고칠 수 없게 된다 |
| **LCEL 체인** `RunnablePassthrough.assign` + `\|` | `fitcast/chains/outfit.py` | 날씨 조회→가이드 수집→LLM 호출을 손으로 순서대로 부르고 dict를 넘겨야 한다. assign은 세 재료를 **병렬**로 준비한다 |
| **구조화 출력** `with_structured_output(OutfitSet)` + Pydantic | `fitcast/schemas.py` | 자유 텍스트를 정규식으로 파싱해야 하고, 아바타에 입힐 `shape`·`color_hex`·검색어를 안정적으로 뽑을 수 없다 |
| **미니 RAG** `MarkdownHeaderTextSplitter` → `init_embeddings` → `InMemoryVectorStore` | `fitcast/rag/style_guide.py` | 12개 스타일 × TPO × 날씨 보정 규칙을 전부 프롬프트에 넣어야 한다(토큰 낭비, 스타일 추가 시 코드 수정) |
| **Tool** `@tool` 4개 (날씨, 가이드 검색, 쇼핑 링크, 실제 상품 검색) | `fitcast/tools/`, `rag/` | LLM이 오늘 날씨를 지어내고, 브랜드·가격을 환각한다 |
| **Tool calling Agent** `create_agent` | `fitcast/agent.py` | "이번 주 토요일 제주도 뭐 입지?"처럼 날짜 계산→날씨 조회→가이드 검색을 상황에 따라 고르는 판단을 코드로 분기해야 한다 |
| **대화 메모리** (history → messages) | `fitcast/agent.py` | "신발만 바꿔줘" 같은 후속 질문을 이해하지 못한다 |

### 3-1. 프롬프트 설계 — 역할과 조건

In [9]:
system_text = OUTFIT_PROMPT.messages[0].prompt.template
print(system_text.split("## shape 코드 가이드")[0])
print("입력 변수:", OUTFIT_PROMPT.input_variables)

너는 날씨와 개인 취향을 함께 고려하는 퍼스널 스타일리스트 'Fitcast'야.

규칙:
1. 날씨의 기온, 체감온도, 강수, 바람, 자외선을 최우선으로 고려한다.
2. 선호 스타일의 분위기를 유지하되, 날씨에 따라 소재, 기장, 두께, 레이어드 수를 조절한다.
3. TPO와 드레스코드에 어긋나는 아이템은 추천하지 않는다.
4. search_keyword는 쇼핑 검색에 바로 사용할 수 있는 짧은 2~3단어 한국어로 작성한다.
5. search_keyword에 브랜드명은 넣지 않는다.
6. 특정 브랜드명, 상품명, 가격, 재고는 지어내지 않는다.
7. 모든 문장은 친근한 한국어 존댓말로 작성한다.
8. 체형 정보가 있으면 체형을 자연스럽게 보완하는 핏과 기장을 선택한다.
9. 체형을 고려한 이유를 각 주요 아이템 설명에 한 번씩 포함한다.
10. shape는 아래 목록에서 해당 부위의 코드만 사용한다.
11. color_hex는 추천 색상과 가까운 실제 HEX 값으로 작성한다.


입력 변수: ['gender', 'note', 'profile', 'shape_guide', 'style_context', 'styles', 'temp_guide', 'tpo', 'weather']


**넣은 것과 근거**

- **역할**: "날씨와 개인 취향을 함께 고려하는 퍼스널 스타일리스트" — 날씨만 보는 기상캐스터도, 취향만 보는 쇼핑 큐레이터도 아니라는 걸 한 문장으로 고정.
- **우선순위 규칙(1·2)**: 날씨 > 스타일. 스타일 가이드(RAG)가 "린넨 원피스"를 권해도 비 오면 조절하라는 뜻. 규칙이 없을 때는 가이드를 그대로 베끼는 결과가 나왔다.
- **금지 규칙(5·6)**: 브랜드·가격·재고 환각 금지. 실제 상품은 Tool(`search_products`)로만 보여주게 분리했다.
- **출력 형식 규칙(4·10·11)**: `search_keyword`는 2~3단어(검색 API에 바로 넣기 위해), `shape`는 **목록에서만**(아바타가 그릴 수 있는 코드만), `color_hex`는 실제 HEX. 스키마의 `description`과 프롬프트 규칙을 **같은 내용으로 이중 명시**해 형식 오류를 줄였다.
- **체형 규칙(8·9)**: 체형이 있으면 보완하는 핏을 고르고 이유에 한 번 언급 — 사용자가 "왜 이 옷?"을 납득하게 하는 서비스 목표.
- **예시(few-shot)는 넣지 않음**: 출력 형식은 Pydantic 스키마의 필드 `description`이 대신 강제하고(3-3), 완성 코디 예시를 넣으면 추천이 예시 조합 쪽으로 쏠릴 위험이 있어 규칙만으로 제약했다. 형식 오류는 스키마 검증에서 잡힌다.
- **데이터는 human 메시지에** `## 날씨 / ## 기온 구간 가이드 / ## 스타일 가이드 / ## 사용자 정보` 헤더로 구분 — 규칙(system)과 매번 바뀌는 컨텍스트(human)를 분리해 프롬프트 파일만 고치면 되게 함.

### 3-2. 체인 구성 — LCEL

In [10]:
from fitcast.chains.outfit import build_outfit_chain

chain = build_outfit_chain()
chain.get_graph().print_ascii()

                     +-----------------------------+                         
                     | Parallel<weather_data>Input |                         
                     +-----------------------------+                         
                              **         ***                                 
                            **              *                                
                           *                 **                              
                    +--------+          +-------------+                      
                    | Lambda |          | Passthrough |                      
                    +--------+          +-------------+                      
                              **         ***                                 
                                **      *                                    
                                  *   **                                     
                    +------------------------------+            

`RunnablePassthrough.assign(...)`은 입력 dict를 그대로 흘리면서 키를 **추가**합니다. 그래서 1단계에서 얻은 `weather_data`를 2단계의 세 함수가 함께 읽고, 3단계 프롬프트는 앞 단계 키를 전부 변수로 받습니다. 2단계의 세 assign은 같은 단계에 있어 **병렬 실행**됩니다(날씨 문장 포맷, 기온 구간표, 임베딩 검색). 마지막 `outfit=OUTFIT_PROMPT | structured_llm`은 프롬프트와 모델을 `|`로 이은 작은 체인입니다.

### 3-3. 구조화 출력 — Pydantic 스키마가 곧 출력 계약

In [11]:
from fitcast.schemas import OutfitSet

schema = OutfitSet.model_json_schema()
for name, prop in schema["$defs"]["OutfitItem"]["properties"].items():
    print(f"OutfitItem.{name:15s} — {prop.get('description', '')}")
print()
for name, prop in schema["properties"].items():
    print(f"OutfitSet.{name:11s} — {prop.get('description', '')}")

OutfitItem.name            — 아이템 이름 (예: 린넨 롱 원피스)
OutfitItem.reason          — 이 날씨·스타일에 이 아이템을 고른 이유 한 문장
OutfitItem.search_keyword  — 쇼핑 검색에 쓸 짧은 2~3단어 한국어 키워드 (색 + 아이템)
OutfitItem.shape           — 아바타에 입힐 모양 코드. 프롬프트의 부위별 목록에서만 고른다
OutfitItem.color_hex       — 아이템 대표 색상 HEX 코드 (예: #1f2a44)

OutfitSet.summary     — 오늘 코디 컨셉 한 줄 요약
OutfitSet.top         — 상의
OutfitSet.bottom      — 하의 (원피스면 원피스를 여기에)
OutfitSet.outer       — 외투 (필요 없으면 null)
OutfitSet.shoes       — 신발
OutfitSet.accessory   — 악세사리·가방·우산 등
OutfitSet.weather_tip — 비·일교차·자외선 등 날씨 관련 주의사항 한 문장


`outer`·`accessory`는 `Optional`이라 "외투가 필요 없는 날"에는 `null`이 옵니다. 파서가 따로 없어도 결과가 바로 Python 객체라서, 웹에서는 `shape`로 아바타 옷을 그리고 `search_keyword`로 상품을 검색하는 후처리가 한 줄씩입니다.

### 3-4. 미니 RAG — 문서 로더·스플리터·임베딩·벡터스토어

In [12]:
from fitcast.rag.style_guide import load_style_docs, get_vectorstore

docs = load_style_docs()
print(f"스타일 가이드 조각: {len(docs)}개 →", [d.metadata["section"] for d in docs])

for q in ["모리걸 더운 날", "결혼식 하객 주의점", "비 오는 날 신발"]:
    hits = get_vectorstore().similarity_search_with_score(q, k=2)
    print(f"\n질의 '{q}' →", [(d.metadata['section'], round(s, 3)) for d, s in hits])

스타일 가이드 조각: 18개 → ['캐주얼', '미니멀', '스트릿', '모리걸', '걸리시', '페미닌', '시티보이', '아메카지', '고프코어', '빈티지', '댄디', '스포티', 'TPO: 출근/오피스', 'TPO: 결혼식 하객', 'TPO: 면접', 'TPO: 여행', 'TPO: 운동/아웃도어', '날씨 보정 규칙']

질의 '모리걸 더운 날' → [('모리걸', 0.498), ('빈티지', 0.439)]



질의 '결혼식 하객 주의점' → [('TPO: 결혼식 하객', 0.412), ('날씨 보정 규칙', 0.231)]

질의 '비 오는 날 신발' → [('캐주얼', 0.413), ('스포티', 0.374)]


`MarkdownHeaderTextSplitter`로 `## 섹션` 단위로 쪼개서 **한 조각 = 한 스타일(또는 TPO)** 이 되게 했습니다. 문단 길이로 자르면 "더운 날/추운 날" 규칙이 다른 조각으로 흩어져 검색이 어긋났습니다. 스타일을 추가할 때는 문서에 섹션 하나만 붙이면 되고 코드는 그대로입니다.

### 3-5. Tool과 Agent, 그리고 대화 메모리

에이전트는 아래 Tool 4개를 상황에 따라 골라 씁니다. 각 Tool의 **docstring이 곧 모델이 읽는 설명**이라, "언제 호출하라"까지 적어 두었습니다.

In [13]:
from fitcast.tools import get_weather, build_shop_links, search_products
from fitcast.rag.style_guide import search_style_guide

for t in (get_weather, search_style_guide, build_shop_links, search_products):
    print(f"● {t.name}({', '.join(t.args)})\n  {t.description.strip().splitlines()[0]}\n")

● get_weather(city, target_date)
  도시의 날씨 예보를 조회한다. 옷차림 추천 전에 반드시 먼저 호출한다.

● search_style_guide(query)
  스타일(모리걸, 미니멀 등)·TPO·날씨 보정 규칙에 대한 내부 가이드를 검색한다.

● build_shop_links(keyword, platforms)
  옷 검색 키워드로 쇼핑 플랫폼별 검색 링크를 만든다. 아이템을 추천할 때마다 호출한다.

● search_products(keyword)
  실제 판매 중인 옷·신발·가방을 찾아야 할 때 호출한다. 판매처·상품명·가격·구매 링크를 돌려준다.



**시연 시나리오**: 1턴 — 날짜 표현("이번 주 토요일")을 스스로 계산해 날씨 Tool을 부르는지 · 2턴 — 이전 대화를 기억해 **신발만** 바꾸는지(메모리).

In [14]:
from fitcast.agent import chat

history = []
q1 = "이번 주 토요일에 제주도 여행 가는데 미니멀하게 뭐 입을까요?"
a1 = chat(q1, history)
display(Markdown(f"**👤 {q1}**\n\n{a1}"))
history += [{"role": "user", "content": q1}, {"role": "assistant", "content": a1}]

**👤 이번 주 토요일에 제주도 여행 가는데 미니멀하게 뭐 입을까요?**

제주도에서의 미니멀한 스타일로 여행을 즐기시려면, 다음과 같은 옷차림을 추천드립니다.

### 추천 복장
- **상의**: 고밀도 코튼 반팔 티셔츠
- **하의**: 린넨 블렌드 슬랙스
- **신발**: 편안한 가죽 샌들이나 로퍼
- **악세사리**: 얇은 실버 주얼리나 가죽 토트백

### 추가 팁
- 일교차가 있을 수 있으니 가벼운 겉옷을 챙기세요.
- 편안한 신발을 선택하셔야 많이 걷는 제주도 여행에 적합합니다.
- 사진 찍을 때 포인트가 될 컬러 아이템을 하나 추가하시면 좋습니다.

이렇게 입으시면 제주도의 날씨와 미니멀한 스타일 모두 잘 어울릴 것입니다! 즐거운 여행 되세요! 🌊✨

In [15]:
q2 = "신발만 좀 더 편한 걸로 바꿔줘"
a2 = chat(q2, history)   # history가 곧 대화 메모리 — 앞 답변을 보고 신발만 바꿔야 한다
display(Markdown(f"**👤 {q2}**\n\n{a2}"))

**👤 신발만 좀 더 편한 걸로 바꿔줘**

제주도의 날씨를 확인해 보니, 24.9°C로 따뜻하고 쾌적한 날씨입니다. 신발을 좀 더 편한 걸로 바꿔드리겠습니다.

### 수정된 추천 복장
- **상의**: 고밀도 코튼 반팔 티셔츠
- **하의**: 린넨 블렌드 슬랙스
- **신발**: 편안한 운동화나 슬리퍼
- **악세사리**: 얇은 실버 주얼리나 가죽 토트백

운동화나 슬리퍼는 걷기에도 편하고 제주도의 자연을 즐기기에 적합합니다. 이렇게 입으시면 더욱 편안하게 여행을 즐기실 수 있을 거예요! 즐거운 여행 되세요! 🌴✨

In [16]:
q3 = "파이썬으로 피보나치 함수 짜줘"
display(Markdown(f"**👤 {q3}**\n\n{chat(q3, history)}"))   # 규칙 12: 패션·날씨 무관 요청은 정중히 거절

**👤 파이썬으로 피보나치 함수 짜줘**

죄송하지만, 프로그래밍 관련 요청은 도와드릴 수 없습니다. 날씨 기반 옷차림이나 패션 관련 질문이 있다면 언제든지 말씀해 주세요!

`create_agent`에 모델·Tool·시스템 프롬프트만 넘기면, 모델이 Tool 호출 → 결과 확인 → 다음 행동을 스스로 반복합니다. 메모리는 별도 컴포넌트 대신 **직전 12개 메시지를 그대로 messages에 넣는** 방식입니다. 시스템 프롬프트에 오늘 날짜를 넣어 "이번 주 토요일"을 계산하게 했고, 날짜가 바뀌면 에이전트를 다시 만듭니다(`lru_cache` 키에 날짜 포함).

위 시연에서 확인한 것: 2턴에서 상의·하의·악세사리는 그대로 두고 **신발만** 바꿨고(메모리), 날씨 Tool을 다시 불러 "24.9°C"를 확인한 뒤 답했다(규칙 1). 3턴은 규칙 12대로 거절했다. 다만 1턴 답변에 규칙 5(아이템 옆 검색 링크)가 지켜지지 않았다 — 4장의 개선점.

---

## 4. 한계와 개선 방향

**테스트에서 드러난 한계**

1. **RAG 오검색** — "TPO: 일상" 질의에 문서에 없는 TPO라서 `TPO: 면접` 조각이 1위로 섞였다(2장 ②). 점수를 보면 관련 조각은 0.4~0.5, 무관한 조각은 0.2~0.4라 **임계값(예: 0.35)** 과 **섹션 종류별 필터**(스타일 질의엔 스타일 조각만)로 걸러낼 수 있다.
2. **검색 키워드 품질** — LLM이 만든 `search_keyword`("베이지 니트 볼레로")가 쇼핑 검색에서 결과 0개인 경우가 있었다. 서비스에서는 색을 떼고("니트 볼레로") 재검색하고, 결과 상품명에 아이템 명사가 있는지로 재정렬하는 폴백을 넣었다. 프롬프트 규칙 4를 "2~3단어"로 고친 것도 이 때문이다.
3. **에이전트가 규칙을 빠뜨림** — 시스템 프롬프트 규칙이 13개로 늘어나자 "아이템마다 검색 링크를 붙여라"(규칙 5)를 건너뛰는 턴이 생겼다. 규칙을 줄이거나, 링크 생성을 모델 판단에 맡기지 않고 **후처리에서 강제**하는 쪽이 안정적이다.
4. **결정성** — 같은 입력이라도 아이템 이름이 실행마다 조금 달라진다. `FITCAST_TEMPERATURE`를 낮추면 안정되지만 추천이 단조로워져, 서비스에서는 기본값을 두고 "랜덤 코디" 버튼으로 분리했다.
5. **환각 방지는 Tool로** — 프롬프트 금지 규칙만으로는 "무신사 ○○ 볼캡 29,000원" 같은 환각을 완전히 막지 못했다. 실제 상품은 `search_products` 결과에 있는 것만 보여주도록 규칙 10·11을 추가하고, 화면에서도 검색 결과 객체만 렌더링하게 했다.
6. **에이전트 지연** — Tool을 3~4번 부르면 답변이 10초를 넘는다. 날씨 조회 결과를 캐시하고, 상품 검색은 사용자가 요청할 때만 부르게 규칙 9로 제한했다.

**다음에 바꿔볼 것**

- 스타일 가이드에 `## TPO: 일상` 섹션을 추가하고, 검색에 **점수 임계값 + 메타데이터 필터**를 넣어 엉뚱한 조각이 섞이는 것 줄이기
- `OutfitItem`에 `alternatives: list[str]`를 추가해 한 슬롯당 대안 2개를 받고, 사용자가 화면에서 바꿔 끼우게 하기
- 사용자가 저장한 코디(회원 DB)를 few-shot 예시로 프롬프트에 넣어 **개인화** 하기
- 이미지 편집 모델로 실제 상품을 아바타에 입히는 "AI 피팅"은 이미 붙어 있으나 40초가 걸려, 결과 캐시와 미리 생성으로 시연 안정성을 확보하기